In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import sklearn
import os
import yfinance as yf
import pandas_datareader.data as web
import time
import requests
import plotly.express as px  # Added for interactive hover
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from google.colab import drive
from datetime import datetime
from scipy.stats import norm
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Setup Standard Directory
base_dir = '/content/drive/MyDrive/MSFin_Python'
sub_folders = ['data', 'notebooks', 'output']

for folder in sub_folders:
    os.makedirs(os.path.join(base_dir, folder), exist_ok=True)

print(f"Environment Ready: Drive mounted and folders verified at {base_dir}")


In [ ]:
# --- CSV FILE NAME FOR THIS ANALYSIS ---
SOURCE_FILENAME = 'L11_Path_114_x1000_REGIME_FWD.csv'

# --- AUTOMATED PATH BUILDING ---
base_name = os.path.splitext(SOURCE_FILENAME)[0]
data_dir = os.path.join(base_dir, 'data')

# Define all file addresses based on the source
input_file = os.path.join(data_dir, SOURCE_FILENAME)
output_file = os.path.join(data_dir, f'L11_Div_Analysis_Results_{base_name}.csv')

print(f"Processing File: {input_file}")
print(f"Results will save to: {os.path.basename(output_file)}")

In [ ]:
def simulate_diversification(df, portfolio_sizes, iterations=30):
    results = []
    all_tickers = df.columns.tolist()

    for size in portfolio_sizes:
        if size > len(all_tickers): continue

        for i in range(iterations):
            # Randomly select 'size' number of paths
            selected_paths = np.random.choice(all_tickers, size=size, replace=False)

            # Create the Portfolio (average of the returns across selected paths)
            portfolio_returns = df[selected_paths].mean(axis=1)

            # Metrics
            ann_vol = portfolio_returns.std() * np.sqrt(12)

            # --- NEW: Cumulative Return Calculation ---
            wealth_index = (1 + portfolio_returns).cumprod()
            cum_ret = wealth_index.iloc[-1] - 1

            # Max Drawdown
            max_dd = ((wealth_index - wealth_index.cummax()) / wealth_index.cummax()).min()

            # Sharpe Ratio (using simple annualized mean for the numerator)
            avg_ret_ann = portfolio_returns.mean() * 12
            sharpe = avg_ret_ann / ann_vol if ann_vol != 0 else 0

            results.append({
                'Portfolio_Size': size,
                'Ann_Volatility': ann_vol,
                'Cumulative_Return': cum_ret,
                'Max_Drawdown': max_dd,
                'Sharpe_Ratio': sharpe
            })

    return pd.DataFrame(results)

# Load the data into df_paths
df_paths = pd.read_csv(input_file)

sizes = [1, 5, 10, 20, 30, 40, 50, 60, 70, 80, 100, 120]
df_div = simulate_diversification(df_paths, sizes)

In [ ]:
# Save results
#div_output_path = os.path.join(data_dir, 'L11_diversification_example.csv')
df_div.to_csv(output_file, index=False)

# Calculate Medians for the specific request
df_medians = df_div.groupby('Portfolio_Size')[['Cumulative_Return', 'Sharpe_Ratio']].median().reset_index()

print("-" * 55)
print("MEDIAN CUMULATIVE RETURN & SHARPE BY PORTFOLIO SIZE")
print("-" * 55)
print(df_medians.to_string(index=False, formatters={
    'Cumulative_Return': '{:,.2%}'.format,
    'Sharpe_Ratio': '{:,.2f}'.format
}))

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_div, x='Portfolio_Size', y='Ann_Volatility', palette='magma')
plt.title('Volatility Decay: Risk Reduction through Path Averaging', fontweight='bold')
plt.ylabel('Annualized Volatility')
plt.gca().yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

color1 = '#1f77b4'
ax1.set_xlabel('Portfolio Size (Number of Paths)')
ax1.set_ylabel('Median Cumulative Return', color=color1, fontweight='bold')
ax1.plot(df_medians['Portfolio_Size'], df_medians['Cumulative_Return'], marker='o', color=color1, lw=3)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))

ax2 = ax1.twinx()
color2 = '#d62728'
ax2.set_ylabel('Median Sharpe Ratio', color=color2, fontweight='bold')
ax2.plot(df_medians['Portfolio_Size'], df_medians['Sharpe_Ratio'], marker='s', color=color2, lw=3)
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('Diversification "Free Lunch": Improving Sharpe, Maintaining Return', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.2)
plt.show()